# Check whether we can run it from ONNX directly or still need to convert to PyTorch

First we check whether ONNX Runtime can even use the GPU

In [1]:
import onnxruntime as ort
print("ORT:", ort.__version__)
print("Available EPs:", ort.get_available_providers())

ORT: 1.23.2
Available EPs: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [2]:
import onnxruntime as ort
import sys

print("Python:", sys.executable)
print("ORT version:", ort.__version__)
print("ORT module file:", ort.__file__)
print("Available EPs:", ort.get_available_providers())

Python: c:\Users\Andreas\miniconda3\envs\das2\python.exe
ORT version: 1.23.2
ORT module file: c:\Users\Andreas\miniconda3\envs\das2\lib\site-packages\onnxruntime\__init__.py
Available EPs: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [3]:
import numpy as np
import onnxruntime as ort

onnx_path = "das_model.onnx"

providers = [
    ("CUDAExecutionProvider", {
        # optional knobs
        "arena_extend_strategy": "kNextPowerOfTwo",
        "do_copy_in_default_stream": 1,
    }),
    "CPUExecutionProvider",
]

sess = ort.InferenceSession(onnx_path, providers=providers)

print("EPs in use:", sess.get_providers())
inp = sess.get_inputs()[0]
out = sess.get_outputs()[0]
print("Input:", inp.name, inp.shape, inp.type)
print("Output:", out.name, out.shape, out.type)

# Use the model's expected shape from sess.get_inputs()[0].shape:
# Example here assumes [B, 8192, 1] or [B, 1024, 1] depending on your exported model.
B = 1
T = inp.shape[1]  # often 8192 or 1024; may be dynamic
C = inp.shape[2]

# If T is dynamic (None / 'unk__'), pick a valid length you trained/exported with.
if isinstance(T, str) or T is None:
    T = 8192  # set to what your model expects

x = np.random.randn(B, T, C).astype(np.float32)
y = sess.run(None, {inp.name: x})[0]
print("y:", y.shape, y.dtype)

EPs in use: ['CPUExecutionProvider']
Input: input_1 ['unk__978', 8192, 1] tensor(float)
Output: up_sampling1d ['unk__979', 'unk__980', 2] tensor(float)
y: (1, 8192, 2) float32
